# Comparison of Candidate JWST CMD Projections

This notebook explores how the geometry of young stellar isochrones depends on the JWST/NIRCam filters used to define a color--magnitude diagram.

The analysis uses the updated merged Baraffe--Pisa--Ekstrom--PARSEC evolution model and BT-Settl 2015 stellar atmospheres. Theoretical isochrones are generated from 1 to 20 Myr at intervals of 0.5 Myr.

The candidate CMDs are divided into four groups:

1. Molecular-feature and water-sensitive combinations
2. Medium--wide hybrid combinations
3. Broad-band continuum controls
4. Long-wavelength controls

For each group, two figures are generated:

- the full theoretical isochrone grid;
- the same simulated cluster snapshot projected into each CMD.

The snapshot panels use the same retained primary-star sample in every projection. This ensures that differences in morphology arise from filter choice rather than differences in which stars were successfully interpolated.

In [1]:
import os
import sys
import math
import warnings
import contextlib
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from astropy.table import Table, Column

from spisea import synthetic, atmospheres, reddening
from nbody6tools import Reader
from nbody62spisea import converter

sys.path.append('/home/wyz5rge/synthetic-cmd-dev/cmd_generator')
import interpolator

/home/wyz5rge/.venv/nbody/lib/python3.11/site-packages/pysynphot/locations.py:345: UserWarning: Extinction files not found in /home/wyz5rge/SPISEA/cdbs/extinction
  warnings.warn('Extinction files not found in %s' % (extdir, ))


particles.bound_subset and particles.bound_indexes will raise ImportError


In [2]:
# ============================================================
# Paths
# ============================================================

UPDATED_MERGED_ROOT = Path(
    '/home/wyz5rge/SPISEA/evolution/merged/'
    'baraffe_pisa_ekstrom_parsec/'
)

SIM_PATH = (
    '/standard/Tan_JC/backup_protoclusters/multiples/'
    'M3000new/sigma0p1/fiducial/sfe_ff003/00'
)

# Save the SPISEA synthetic-photometry cache locally beside the notebook.
NOTEBOOK_DIR = Path.cwd()

ISO_CACHE_DIR = (
    NOTEBOOK_DIR /
    'isochrones_merged_nircam_filter_exploration'
)

# Set this to True only when you deliberately want to regenerate
# every cached photometric isochrone from scratch.
RESET_ISO_CACHE = False

if RESET_ISO_CACHE and ISO_CACHE_DIR.exists():
    shutil.rmtree(ISO_CACHE_DIR)

ISO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook directory:')
print(NOTEBOOK_DIR)

print('\nIsochrone cache directory:')
print(ISO_CACHE_DIR)


# ============================================================
# Model and observational settings
# ============================================================

USE_ROTATING_MERGED = False

AKs = 0.0
dist = 410
metallicity = 0.0

atm_func = atmospheres.get_BTSettl_2015_atmosphere
red_law = reddening.RedLawHosek18b()


# ============================================================
# Theoretical isochrone age grid
# ============================================================

AGE_MIN_MYR = 1.0
AGE_MAX_MYR = 20.0
AGE_STEP_MYR = 0.5

ISO_AGES_MYR = np.arange(
    AGE_MIN_MYR,
    AGE_MAX_MYR + 0.5 * AGE_STEP_MYR,
    AGE_STEP_MYR
)

LOG_AGES = np.log10(ISO_AGES_MYR * 1e6)

print('\nNumber of theoretical isochrones:', len(ISO_AGES_MYR))
print('Age range:', ISO_AGES_MYR[0], 'to', ISO_AGES_MYR[-1], 'Myr')

Notebook directory:
/sfs/gpfs/tardis/home/wyz5rge/synthetic-cmd-dev/notebooks/2026/07-14_02_model-and-filter-selection

Isochrone cache directory:
/sfs/gpfs/tardis/home/wyz5rge/synthetic-cmd-dev/notebooks/2026/07-14_02_model-and-filter-selection/isochrones_merged_nircam_filter_exploration

Number of theoretical isochrones: 39
Age range: 1.0 to 20.0 Myr


In [3]:
# Generate all candidate filters simultaneously.
ALL_FILTERS = [
    'jwst,F090W',
    'jwst,F115W',
    'jwst,F140M',
    'jwst,F150W',
    'jwst,F162M',
    'jwst,F182M',
    'jwst,F200W',
    'jwst,F210M',
    'jwst,F277W',
    'jwst,F356W',
    'jwst,F444W',
]

FILTER_KEYS = {
    'F090W': 'm_jwst_F090W',
    'F115W': 'm_jwst_F115W',
    'F140M': 'm_jwst_F140M',
    'F150W': 'm_jwst_F150W',
    'F162M': 'm_jwst_F162M',
    'F182M': 'm_jwst_F182M',
    'F200W': 'm_jwst_F200W',
    'F210M': 'm_jwst_F210M',
    'F277W': 'm_jwst_F277W',
    'F356W': 'm_jwst_F356W',
    'F444W': 'm_jwst_F444W',
}

print('Requested filters:')
for filt in ALL_FILTERS:
    print(' ', filt)

Requested filters:
  jwst,F090W
  jwst,F115W
  jwst,F140M
  jwst,F150W
  jwst,F162M
  jwst,F182M
  jwst,F200W
  jwst,F210M
  jwst,F277W
  jwst,F356W
  jwst,F444W


In [4]:
class MergedBaraffePisaEkstromParsecDAT:
    """
    Compatibility reader for the merged
    Baraffe--Pisa--Ekstrom--PARSEC model stored as ASCII .dat files.

    The root directory must contain:
        z015_rot/
        z015_norot/
    """

    def __init__(self, root_dir, rot=False):
        self.root_dir = Path(root_dir).expanduser().resolve()
        self.rot = bool(rot)

        self.model_dir = str(self.root_dir)
        self.z_list = [0.015]
        self.z_solar = 0.015
        self.mass_list = []

        self.grid_dir = self.root_dir / (
            'z015_rot' if self.rot else 'z015_norot'
        )

        if not self.grid_dir.is_dir():
            raise FileNotFoundError(
                f'Merged-model directory not found: {self.grid_dir}'
            )

        files = sorted(self.grid_dir.glob('iso_*.dat'))

        if len(files) == 0:
            raise FileNotFoundError(
                f'No iso_*.dat files found in {self.grid_dir}'
            )

        self.age_file_map = {}

        for path in files:
            try:
                log_age = float(path.stem.split('_')[1])
            except (IndexError, ValueError):
                continue

            self.age_file_map[round(log_age, 2)] = path

        self.age_list = np.array(
            sorted(self.age_file_map.keys()),
            dtype=float
        )

    def isochrone(self, age=1.0e6, metallicity=0.0):
        log_age_requested = math.log10(age)

        if log_age_requested < self.age_list[0]:
            raise ValueError(
                f'Requested logAge={log_age_requested:.4f} is younger '
                f'than grid minimum logAge={self.age_list[0]:.2f}.'
            )

        if log_age_requested > self.age_list[-1]:
            raise ValueError(
                f'Requested logAge={log_age_requested:.4f} is older '
                f'than grid maximum logAge={self.age_list[-1]:.2f}.'
            )

        # Select the nearest native age file.
        idx = np.argmin(
            np.abs(self.age_list - log_age_requested)
        )
        selected_log_age = float(self.age_list[idx])

        iso_path = self.age_file_map[
            round(selected_log_age, 2)
        ]

        dtype = [
            ('mass', 'f8'),
            ('logT', 'f8'),
            ('logL', 'f8'),
            ('logg', 'f8'),
            ('logT_WR', 'f8'),
            ('mass_current', 'f8'),
            ('phase', 'i4'),
            ('model_ref', 'U32'),
        ]

        data = np.genfromtxt(
            str(iso_path),
            comments='#',
            dtype=dtype,
            encoding='utf-8'
        )

        data = np.atleast_1d(data)
        iso = Table(data)

        is_wr = ~np.isclose(
            np.asarray(iso['logT'], dtype=float),
            np.asarray(iso['logT_WR'], dtype=float),
            rtol=0.0,
            atol=1e-8
        )

        iso.add_column(
            Column(is_wr, name='isWR')
        )

        iso.meta['log_age'] = selected_log_age
        iso.meta['log_age_requested'] = log_age_requested
        iso.meta['metallicity_in'] = metallicity
        iso.meta['metallicity_act'] = 0.0
        iso.meta['source_file'] = str(iso_path)

        return iso

In [5]:
evo_model = MergedBaraffePisaEkstromParsecDAT(
    UPDATED_MERGED_ROOT,
    rot=USE_ROTATING_MERGED
)

print('Model class:')
print(type(evo_model))

print('\nGrid directory:')
print(evo_model.grid_dir)

print(
    '\nAvailable native log-age range:',
    evo_model.age_list.min(),
    'to',
    evo_model.age_list.max()
)

test_evolution_iso = evo_model.isochrone(
    age=1.0e6,
    metallicity=metallicity
)

print('\nTest file:')
print(test_evolution_iso.meta['source_file'])

print(
    '\nPhysical mass range at 1 Myr:',
    np.min(test_evolution_iso['mass']),
    'to',
    np.max(test_evolution_iso['mass']),
    'Msun'
)

Model class:
<class '__main__.MergedBaraffePisaEkstromParsecDAT'>

Grid directory:
/sfs/gpfs/tardis/home/wyz5rge/SPISEA/evolution/merged/baraffe_pisa_ekstrom_parsec/z015_norot

Available native log-age range: 6.0 to 10.09

Test file:
/sfs/gpfs/tardis/home/wyz5rge/SPISEA/evolution/merged/baraffe_pisa_ekstrom_parsec/z015_norot/iso_6.00.dat

Physical mass range at 1 Myr: 0.01 to 500.0 Msun


In [6]:
# Each tuple contains:
#   first filter, second filter, y-axis filter, panel title

MOLECULAR_GROUP = [
    ('F140M', 'F162M', 'F162M', 'F140M-F162M vs F162M'),
    ('F162M', 'F182M', 'F162M', 'F162M-F182M vs F162M'),
    ('F162M', 'F210M', 'F210M', 'F162M-F210M vs F210M'),
    ('F182M', 'F210M', 'F210M', 'F182M-F210M vs F210M'),
]

HYBRID_GROUP = [
    ('F140M', 'F200W', 'F200W', 'F140M-F200W vs F200W'),
    ('F162M', 'F200W', 'F200W', 'F162M-F200W vs F200W'),
    ('F182M', 'F200W', 'F200W', 'F182M-F200W vs F200W'),
    ('F200W', 'F210M', 'F210M', 'F200W-F210M vs F210M'),
]

BROADBAND_GROUP = [
    ('F090W', 'F150W', 'F150W', 'F090W-F150W vs F150W'),
    ('F090W', 'F200W', 'F200W', 'F090W-F200W vs F200W'),
    ('F115W', 'F150W', 'F150W', 'F115W-F150W vs F150W'),
    ('F115W', 'F200W', 'F200W', 'F115W-F200W vs F200W'),
    ('F150W', 'F200W', 'F200W', 'F150W-F200W vs F200W'),
]

LONG_WAVELENGTH_GROUP = [
    ('F200W', 'F277W', 'F277W', 'F200W-F277W vs F277W'),
    ('F200W', 'F356W', 'F356W', 'F200W-F356W vs F356W'),
    ('F277W', 'F356W', 'F356W', 'F277W-F356W vs F356W'),
    ('F356W', 'F444W', 'F444W', 'F356W-F444W vs F444W'),
]

CMD_GROUPS = {
    'Molecular-feature and H2O-sensitive colors': MOLECULAR_GROUP,
    'Medium-wide hybrid colors': HYBRID_GROUP,
    'Broad-band continuum controls': BROADBAND_GROUP,
    'Long-wavelength controls': LONG_WAVELENGTH_GROUP,
}

for group_name, group in CMD_GROUPS.items():
    print('\n' + group_name)
    for item in group:
        print(' ', item[3])


Molecular-feature and H2O-sensitive colors
  F140M-F162M vs F162M
  F162M-F182M vs F162M
  F162M-F210M vs F210M
  F182M-F210M vs F210M

Medium-wide hybrid colors
  F140M-F200W vs F200W
  F162M-F200W vs F200W
  F182M-F200W vs F200W
  F200W-F210M vs F210M

Broad-band continuum controls
  F090W-F150W vs F150W
  F090W-F200W vs F200W
  F115W-F150W vs F150W
  F115W-F200W vs F200W
  F150W-F200W vs F200W

Long-wavelength controls
  F200W-F277W vs F277W
  F200W-F356W vs F356W
  F277W-F356W vs F356W
  F356W-F444W vs F444W


In [ ]:
def build_multifilter_isochrone_grid():
    iso_grid = []
    records = []

    for age_myr, log_age in zip(ISO_AGES_MYR, LOG_AGES):
        print(f'Building age = {age_myr:.1f} Myr')

        try:
            iso = synthetic.IsochronePhot(
                log_age,
                AKs,
                dist,
                metallicity=metallicity,
                evo_model=evo_model,
                atm_func=atm_func,
                red_law=red_law,
                filters=ALL_FILTERS,
                iso_dir=str(ISO_CACHE_DIR)
            )

            missing = [
                key for key in FILTER_KEYS.values()
                if key not in iso.points.colnames
            ]

            if missing:
                raise KeyError(
                    'Photometric isochrone is missing columns: '
                    f'{missing}\n'
                    'Use a fresh cache directory or set '
                    'RESET_ISO_CACHE=True and rerun.'
                )

            mass = np.asarray(
                iso.points['mass'],
                dtype=float
            )

            iso_grid.append(iso)

            records.append({
                'age_myr': age_myr,
                'log_age': log_age,
                'n_points': len(iso.points),
                'mass_min': np.nanmin(mass),
                'mass_max': np.nanmax(mass),
                'status': 'success',
                'error': '',
            })

        except Exception as exc:
            print(f'FAILED at {age_myr:.1f} Myr: {exc}')

            iso_grid.append(None)

            records.append({
                'age_myr': age_myr,
                'log_age': log_age,
                'n_points': 0,
                'mass_min': np.nan,
                'mass_max': np.nan,
                'status': 'failed',
                'error': str(exc),
            })

    return iso_grid, pd.DataFrame(records)


ISO_GRID, df_iso_coverage = build_multifilter_isochrone_grid()

display(df_iso_coverage)

Building age = 1.0 Myr


In [ ]:
successful_isochrones = df_iso_coverage[
    df_iso_coverage['status'] == 'success'
]

print(
    'Successful theoretical isochrones:',
    len(successful_isochrones),
    '/',
    len(df_iso_coverage)
)

if len(successful_isochrones) > 0:
    print(
        'Photometric mass range represented across the grid:',
        successful_isochrones['mass_min'].min(),
        'to',
        successful_isochrones['mass_max'].max(),
        'Msun'
    )

failed_isochrones = df_iso_coverage[
    df_iso_coverage['status'] != 'success'
]

if len(failed_isochrones) > 0:
    print('\nFailed ages:')
    display(failed_isochrones)

In [ ]:
L_SUN_WATTS = 3.846e26


def get_hr_coordinates(iso):
    if iso is None:
        return np.array([]), np.array([]), np.array([])

    pts = iso.points

    teff = np.asarray(pts['Teff'], dtype=float)
    luminosity_watts = np.asarray(pts['L'], dtype=float)
    mass = np.asarray(pts['mass'], dtype=float)

    log_luminosity = np.log10(
        luminosity_watts / L_SUN_WATTS
    )

    good = (
        np.isfinite(teff) &
        np.isfinite(log_luminosity) &
        np.isfinite(mass)
    )

    return teff[good], log_luminosity[good], mass[good]


def get_cmd_coordinates(iso, cmd_choice):
    if iso is None:
        return np.array([]), np.array([]), np.array([])

    filt_a, filt_b, y_filter, _ = cmd_choice

    pts = iso.points

    key_a = FILTER_KEYS[filt_a]
    key_b = FILTER_KEYS[filt_b]
    key_y = FILTER_KEYS[y_filter]

    mag_a = np.asarray(pts[key_a], dtype=float)
    mag_b = np.asarray(pts[key_b], dtype=float)
    ymag = np.asarray(pts[key_y], dtype=float)
    mass = np.asarray(pts['mass'], dtype=float)

    color = mag_a - mag_b

    good = (
        np.isfinite(color) &
        np.isfinite(ymag) &
        np.isfinite(mass)
    )

    return color[good], ymag[good], mass[good]


def add_age_colorbar(fig, axes):
    cmap = plt.get_cmap('coolwarm')
    norm = Normalize(
        vmin=ISO_AGES_MYR.min(),
        vmax=ISO_AGES_MYR.max()
    )

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=np.asarray(axes).ravel().tolist(),
        pad=0.015,
        fraction=0.025
    )

    cbar.set_label('Isochrone age (Myr)')

    return cmap, norm

In [ ]:
def plot_theoretical_group(
    group_name,
    cmd_choices,
    include_hr=True
):
    n_panels = len(cmd_choices) + int(include_hr)

    ncols = 3
    nrows = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(16, 5.0 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    cmap = plt.get_cmap('coolwarm')
    norm = Normalize(
        vmin=ISO_AGES_MYR.min(),
        vmax=ISO_AGES_MYR.max()
    )

    panel_idx = 0

    if include_hr:
        ax = axes[panel_idx]

        for iso, age_myr in zip(ISO_GRID, ISO_AGES_MYR):
            x, y, _ = get_hr_coordinates(iso)

            if len(x) == 0:
                continue

            ax.plot(
                x,
                y,
                color=cmap(norm(age_myr)),
                lw=1.0,
                alpha=0.68
            )

        ax.set_title(r'Luminosity vs $T_{\rm eff}$')
        ax.set_xlabel(r'$T_{\rm eff}$ (K)')
        ax.set_ylabel(r'$\log_{10}(L/L_\odot)$')
        ax.invert_xaxis()
        ax.grid(alpha=0.22)

        panel_idx += 1

    for cmd_choice in cmd_choices:
        ax = axes[panel_idx]

        filt_a, filt_b, y_filter, title = cmd_choice

        for iso, age_myr in zip(ISO_GRID, ISO_AGES_MYR):
            x, y, _ = get_cmd_coordinates(
                iso,
                cmd_choice
            )

            if len(x) == 0:
                continue

            ax.plot(
                x,
                y,
                color=cmap(norm(age_myr)),
                lw=1.0,
                alpha=0.68
            )

        ax.set_title(title)
        ax.set_xlabel(f'{filt_a} - {filt_b}')
        ax.set_ylabel(y_filter)
        ax.invert_yaxis()
        ax.grid(alpha=0.22)

        panel_idx += 1

    for ax in axes[panel_idx:]:
        ax.axis('off')

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes[:panel_idx].tolist(),
        pad=0.015,
        fraction=0.025
    )
    cbar.set_label('Isochrone age (Myr)')

    fig.suptitle(
        f'{group_name}\n'
        'Merged-model theoretical isochrones, 1-20 Myr',
        fontsize=16
    )

    plt.subplots_adjust(
        top=0.90,
        right=0.91,
        wspace=0.28,
        hspace=0.31
    )

    plt.show()

## Group 1: Molecular-feature and H2O-sensitive colors

These combinations use medium-band filters that sample different portions of the near-infrared spectra of cool stars. F162M acts approximately as an off-band reference for the strong molecular structure surrounding the F182M region, while F140M, F182M, and F210M provide alternative measurements across molecularly structured portions of the spectrum.

### Potential advantages

- Strong color sensitivity to effective temperature in very cool, low-mass stars
- Potentially improved separation of objects near and below the IMF peak
- Direct relevance to the F162M/F182M strategy used in low-mass and low-metallicity IMF studies
- Useful for determining whether molecular absorption is responsible for the pronounced bends seen in some CMDs

### Potential disadvantages

- Molecular structure can make color non-monotonic with mass or effective temperature
- Isochrones may develop hooks, severe curvature, or self-overlap
- A single color can correspond to multiple magnitudes or masses
- Simple vertical binning may mix physically different portions of the stellar sequence

These panels test whether the increased low-mass sensitivity compensates for the added geometric complexity.

In [ ]:
plot_theoretical_group(
    'Molecular-feature and H2O-sensitive colors',
    MOLECULAR_GROUP,
    include_hr=True
)

## Group 2: Medium--wide hybrid colors

These combinations pair a molecularly sensitive medium filter with a broader continuum-like band, usually F200W. They are intended to preserve some of the temperature sensitivity of the medium filters while smoothing over the strongest narrow spectral structure.

### Potential advantages

- Retains sensitivity to cool-star molecular absorption
- Wider color baselines may provide greater separation between stellar masses
- Broad y-axis filters generally yield high throughput and observational practicality
- May produce smoother and more nearly single-valued CMD loci than medium--medium colors
- F182M-F200W versus F200W is a particularly promising compromise between temperature sensitivity and geometric regularity

### Potential disadvantages

- Some molecularly induced curvature may remain
- A longer color baseline can increase sensitivity to extinction
- The age-separation direction may not remain vertical across the complete sequence
- F200W and F210M are close enough in wavelength that F200W-F210M may provide limited color leverage

These panels are likely to contain the strongest candidates for the final ensemble analysis.

In [ ]:
plot_theoretical_group(
    'Medium-wide hybrid colors',
    HYBRID_GROUP,
    include_hr=True
)

## Group 3: Broad-band continuum controls

These projections use broad NIRCam filters that average over comparatively large wavelength intervals. They provide a control sample for determining whether severe CMD curvature is specifically associated with medium-band molecular features.

### Potential advantages

- Higher throughput than medium filters
- Less sensitivity to individual molecular absorption bands
- Potentially smoother and more monotonic mass-to-color relations
- More straightforward observational application
- Useful for testing whether the geometric artifacts disappear when molecular structure is averaged over

### Potential disadvantages

- Reduced sensitivity to the detailed spectra of the coolest stars
- Low-mass stars may occupy a compressed color range
- Broad colors can be more sensitive to extinction and reddening
- A smooth CMD does not necessarily provide strong vertical age separation

F150W-F200W versus F200W is an especially useful control because it spans the same general short-wavelength region without explicitly isolating the F182M molecular feature.

In [ ]:
plot_theoretical_group(
    'Broad-band continuum controls',
    BROADBAND_GROUP,
    include_hr=True
)

## Group 4: Long-wavelength controls

These CMDs extend into the NIRCam long-wavelength channel. They probe a different part of the stellar spectral-energy distribution and provide a test of whether favorable age-spread geometry exists outside the short-wavelength molecular-feature regime.

### Potential advantages

- Large wavelength baselines can produce substantial color separation
- Reduced extinction relative to shorter wavelengths
- F200W-F356W is observationally relevant to existing JWST cluster data
- Provides an independent check on whether conclusions depend specifically on the F162M/F182M region

### Potential disadvantages

- Photospheric age sensitivity may be weaker at longer wavelengths
- Real young clusters may contain circumstellar-disk or dust excess in these bands
- Long-wavelength photometry may introduce astrophysical scatter unrelated to stellar age
- Wide-baseline colors may be strongly affected by reddening, calibration, and atmosphere-model choices

These projections are best treated as controls unless they show exceptionally favorable theoretical geometry.

In [ ]:
plot_theoretical_group(
    'Long-wavelength controls',
    LONG_WAVELENGTH_GROUP,
    include_hr=True
)

# Projection of a Common Simulation Snapshot

The following figures project one simulated cluster snapshot into the same four groups of CMDs.

Only primary stars are included in this initial geometric comparison. A star is retained only if:

1. its age is within the 1--20 Myr theoretical grid;
2. its mass lies within the usable photometric isochrone range;
3. interpolation succeeds for every required filter.

The final catalog is therefore a common retained sample used in every panel. This prevents apparent differences between CMDs from being caused by different stars being accepted in different filter combinations.

Stars younger than 1 Myr are excluded because the merged evolution grid begins at 1 Myr. Their exclusion is reported explicitly below.

In [ ]:
SNAPSHOT_TIME_MYR = 1.5


def load_cluster_table(sim_path, snapshot_time_myr):
    sim_path = os.path.abspath(sim_path)

    if not sim_path.endswith('/'):
        sim_path += '/'

    print('Reading snapshot from:')
    print(sim_path)

    snapshot = Reader.read_snapshot(
        sim_path,
        time=snapshot_time_myr
    )

    snapshot.to_physical()

    return converter.to_spicea_table(snapshot)


cluster_table = load_cluster_table(
    SIM_PATH,
    SNAPSHOT_TIME_MYR
)

print('\nNumber of simulated systems:', len(cluster_table))
print('Snapshot time requested:', SNAPSHOT_TIME_MYR, 'Myr')

In [ ]:
def safe_interpolate(
    age_myr,
    mass,
    iso_grid,
    log_age_arr,
    filter_pair
):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')

            with contextlib.redirect_stdout(io.StringIO()):
                with contextlib.redirect_stderr(io.StringIO()):
                    result = interpolator.interpolate(
                        age_myr,
                        mass,
                        iso_grid,
                        log_age_arr,
                        filter_pair
                    )

        if result is None:
            return None

        result = np.asarray(result, dtype=float)

        if not np.all(np.isfinite(result)):
            return None

        return result

    except Exception:
        return None

In [ ]:
# Filter pairs sufficient to recover every requested magnitude.
INTERPOLATION_PAIRS = [
    ('F090W', 'F115W'),
    ('F140M', 'F150W'),
    ('F162M', 'F182M'),
    ('F200W', 'F210M'),
    ('F277W', 'F356W'),
    ('F356W', 'F444W'),
]


def build_common_snapshot_catalog(
    cluster_table,
    iso_grid,
    log_age_arr
):
    masses = np.asarray(
        cluster_table['mass'],
        dtype=float
    )

    ages_myr = np.asarray(
        cluster_table['age'],
        dtype=float
    )

    grid_ages_myr = (
        np.power(10.0, np.asarray(log_age_arr, dtype=float))
        / 1e6
    )

    rows = []

    rejection_counts = {
        'age_below_grid': 0,
        'age_above_grid': 0,
        'interpolation_failure': 0,
        'retained': 0,
    }

    for system_index, (mass, age_myr) in enumerate(
        zip(masses, ages_myr)
    ):
        if age_myr < grid_ages_myr[0]:
            rejection_counts['age_below_grid'] += 1
            continue

        if age_myr > grid_ages_myr[-1]:
            rejection_counts['age_above_grid'] += 1
            continue

        row = {
            'system_index': system_index,
            'mass': mass,
            'age_myr': age_myr,
        }

        successful = True
        physical_values_stored = False

        for filt_a, filt_b in INTERPOLATION_PAIRS:
            key_a = FILTER_KEYS[filt_a]
            key_b = FILTER_KEYS[filt_b]

            star = safe_interpolate(
                age_myr,
                mass,
                iso_grid,
                log_age_arr,
                [key_a, key_b]
            )

            if star is None:
                successful = False
                break

            # Interpolator output:
            # [luminosity, Teff, logg, mag_a, mag_b]
            luminosity_watts = float(star[0])
            teff = float(star[1])
            logg = float(star[2])
            mag_a = float(star[3])
            mag_b = float(star[4])

            if not physical_values_stored:
                row['luminosity_watts'] = luminosity_watts
                row['log_luminosity_lsun'] = np.log10(
                    luminosity_watts / L_SUN_WATTS
                )
                row['teff'] = teff
                row['logg'] = logg
                physical_values_stored = True

            row[f'mag_{filt_a}'] = mag_a
            row[f'mag_{filt_b}'] = mag_b

        if not successful:
            rejection_counts['interpolation_failure'] += 1
            continue

        rows.append(row)
        rejection_counts['retained'] += 1

    return pd.DataFrame(rows), rejection_counts


df_snapshot, snapshot_rejections = build_common_snapshot_catalog(
    cluster_table,
    ISO_GRID,
    LOG_AGES
)

print('Snapshot retention results:')

for reason, count in snapshot_rejections.items():
    print(f'  {reason}: {count}')

print('\nCommon retained primary-star sample:', len(df_snapshot))

display(df_snapshot.head())

In [ ]:
required_snapshot_columns = [
    f'mag_{filt_name}'
    for filt_name in FILTER_KEYS
]

missing_snapshot_columns = [
    col for col in required_snapshot_columns
    if col not in df_snapshot.columns
]

if missing_snapshot_columns:
    raise KeyError(
        'Snapshot catalog is missing columns: '
        f'{missing_snapshot_columns}'
    )

print('All required filter magnitudes are present.')

In [ ]:
def plot_snapshot_group(
    group_name,
    cmd_choices,
    include_hr=True
):
    n_panels = len(cmd_choices) + int(include_hr)

    ncols = 3
    nrows = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(16, 5.0 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    cmap = plt.get_cmap('coolwarm')
    norm = Normalize(
        vmin=ISO_AGES_MYR.min(),
        vmax=ISO_AGES_MYR.max()
    )

    panel_idx = 0

    if include_hr:
        ax = axes[panel_idx]

        # Isochrones
        for iso, age_myr in zip(ISO_GRID, ISO_AGES_MYR):
            x_iso, y_iso, _ = get_hr_coordinates(iso)

            if len(x_iso) == 0:
                continue

            ax.plot(
                x_iso,
                y_iso,
                color=cmap(norm(age_myr)),
                lw=0.8,
                alpha=0.25,
                zorder=1
            )

        # Simulation stars
        ax.scatter(
            df_snapshot['teff'],
            df_snapshot['log_luminosity_lsun'],
            s=7,
            color='black',
            alpha=0.50,
            edgecolors='none',
            zorder=3
        )

        ax.set_title(r'Luminosity vs $T_{\rm eff}$')
        ax.set_xlabel(r'$T_{\rm eff}$ (K)')
        ax.set_ylabel(r'$\log_{10}(L/L_\odot)$')
        ax.invert_xaxis()
        ax.grid(alpha=0.22)

        panel_idx += 1

    for cmd_choice in cmd_choices:
        ax = axes[panel_idx]

        filt_a, filt_b, y_filter, title = cmd_choice

        # Theoretical isochrones
        for iso, age_myr in zip(ISO_GRID, ISO_AGES_MYR):
            x_iso, y_iso, _ = get_cmd_coordinates(
                iso,
                cmd_choice
            )

            if len(x_iso) == 0:
                continue

            ax.plot(
                x_iso,
                y_iso,
                color=cmap(norm(age_myr)),
                lw=0.8,
                alpha=0.25,
                zorder=1
            )

        # Common simulation sample
        color = (
            df_snapshot[f'mag_{filt_a}']
            - df_snapshot[f'mag_{filt_b}']
        )

        ymag = df_snapshot[f'mag_{y_filter}']

        ax.scatter(
            color,
            ymag,
            s=7,
            color='black',
            alpha=0.50,
            edgecolors='none',
            zorder=3
        )

        ax.set_title(title)
        ax.set_xlabel(f'{filt_a} - {filt_b}')
        ax.set_ylabel(y_filter)
        ax.invert_yaxis()
        ax.grid(alpha=0.22)

        panel_idx += 1

    for ax in axes[panel_idx:]:
        ax.axis('off')

    for ax in axes[:panel_idx]:
        ax.text(
            0.03,
            0.97,
            f'N = {len(df_snapshot)}',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=9,
            bbox=dict(
                boxstyle='round',
                facecolor='white',
                alpha=0.82
            )
        )

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes[:panel_idx].tolist(),
        pad=0.015,
        fraction=0.025
    )
    cbar.set_label('Isochrone age (Myr)')

    fig.suptitle(
        f'{group_name}\n'
        rf'Common cluster snapshot: $\Sigma_{{\rm cloud}}=0.1$ '
        rf'g cm$^{{-2}}$, $\epsilon_{{\rm ff}}=0.03$, '
        f't={SNAPSHOT_TIME_MYR:.2f} Myr',
        fontsize=16
    )

    plt.subplots_adjust(
        top=0.89,
        right=0.91,
        wspace=0.28,
        hspace=0.31
    )

    plt.show()

In [ ]:
plot_snapshot_group(
    'Molecular-feature and H2O-sensitive colors',
    MOLECULAR_GROUP,
    include_hr=True
)

In [ ]:
plot_snapshot_group(
    'Medium-wide hybrid colors',
    HYBRID_GROUP,
    include_hr=True
)

In [ ]:
plot_snapshot_group(
    'Broad-band continuum controls',
    BROADBAND_GROUP,
    include_hr=True
)

In [ ]:
plot_snapshot_group(
    'Long-wavelength controls',
    LONG_WAVELENGTH_GROUP,
    include_hr=True
)

In [ ]:
plot_snapshot_group(
    'Long-wavelength controls',
    LONG_WAVELENGTH_GROUP,
    include_hr=True
)

In [ ]:
range_rows = []

for group_name, group in CMD_GROUPS.items():
    for filt_a, filt_b, y_filter, title in group:
        color = (
            df_snapshot[f'mag_{filt_a}']
            - df_snapshot[f'mag_{filt_b}']
        )

        ymag = df_snapshot[f'mag_{y_filter}']

        range_rows.append({
            'group': group_name,
            'diagram': title,
            'color_min': np.nanmin(color),
            'color_max': np.nanmax(color),
            'color_range': np.nanmax(color) - np.nanmin(color),
            'ymag_min': np.nanmin(ymag),
            'ymag_max': np.nanmax(ymag),
            'ymag_range': np.nanmax(ymag) - np.nanmin(ymag),
        })

df_panel_ranges = pd.DataFrame(range_rows)

display(df_panel_ranges)

In [ ]:
LOW_MASS_MIN = 0.01
LOW_MASS_MAX = 1.4


def get_cmd_coordinates_mass_limited(
    iso,
    cmd_choice,
    mass_min=LOW_MASS_MIN,
    mass_max=LOW_MASS_MAX
):
    x, y, mass = get_cmd_coordinates(
        iso,
        cmd_choice
    )

    keep = (
        (mass >= mass_min) &
        (mass <= mass_max)
    )

    return x[keep], y[keep], mass[keep]


def get_hr_coordinates_mass_limited(
    iso,
    mass_min=LOW_MASS_MIN,
    mass_max=LOW_MASS_MAX
):
    x, y, mass = get_hr_coordinates(iso)

    keep = (
        (mass >= mass_min) &
        (mass <= mass_max)
    )

    return x[keep], y[keep], mass[keep]